# 03. Customer Cohort & 90-Day Repeat Label Validation

## Objective

Construct a customer-level cohort from payment-approved orders and validate the 90-day repeat-purchase outcome.

The validation focuses on:

- one row per customer
- first approved purchase
- complete 90-day observation window
- repeat-purchase label integrity
- baseline repeat-purchase prevalence

In [1]:
import sqlite3

import pandas as pd

In [2]:
conn = sqlite3.connect(
    "../data/processed/olist.db"
)

customer_df = pd.read_sql_query(
    """
    SELECT *
    FROM customer_repeat_90d_base
    """,
    conn
)

conn.close()

In [3]:
# Customer Grain 확인
print(
    "rows:",
    len(customer_df)
)

print(
    "unique customers:",
    customer_df[
        "customer_unique_id"
    ]
    .nunique()
)

print(
    "duplicate customers:",
    customer_df[
        "customer_unique_id"
    ]
    .duplicated()
    .sum()
)

rows: 95997
unique customers: 95997
duplicate customers: 0


### Finding — Customer-Level Grain

The cohort contains one row per customer_unique_id.

Each customer's first approved purchase defines the prediction snapshot for the 90-day outcome.

In [4]:
# 관찰 가능/불가능 고객 확인
print(
    customer_df[
        "is_eligible_90d"
    ]
    .value_counts(
        dropna=False
    )
)

print()

print(
    "Reference Target"
)

print(
    customer_df[
        "repeat_90d"
    ]
    .value_counts(
        dropna=False
    )
)

print()

print(
    "Primary Target"
)

print(
    customer_df[
        "repeat_90d_after_1h"
    ]
    .value_counts(
        dropna=False
    )
)

is_eligible_90d
1    78505
0    17492
Name: count, dtype: int64

Reference Target
repeat_90d
0.0    76739
NaN    17492
1.0     1766
Name: count, dtype: int64

Primary Target
repeat_90d_after_1h
0.0    77423
NaN    17492
1.0     1082
Name: count, dtype: int64


### Finding — Outcome Eligibility

Customers whose first purchase occurs after the 90-day eligibility cutoff are not assigned a negative label.

Their repeat_90d value remains missing because the full outcome window is not observable.

In [5]:
# Baseline 재구매율
eligible_customer_df = (
    customer_df[
        customer_df[
            "is_eligible_90d"
        ] == 1
    ]
)

repeat_rate = (
    eligible_customer_df[
        "repeat_90d"
    ]
    .mean()
)

print(
    f"90-day repeat rate: {repeat_rate:.2%}"
)

primary_repeat_rate = (
    eligible_customer_df[
        "repeat_90d_after_1h"
    ]
    .mean()
)

print(
    f"Primary 90-day repeat rate: "
    f"{primary_repeat_rate:.2%}"
)

90-day repeat rate: 2.25%
Primary 90-day repeat rate: 1.38%


In [6]:
# Sample Audit

## repeat 고객
repeat_sample_df = (
    customer_df[
        customer_df[
            "repeat_90d_after_1h"
        ] == 1
    ]
    .head(5)
)

repeat_sample_df[
    [
        "customer_unique_id",
        "first_approved_at",
        "second_approved_at_after_1h",
        "repeat_90d_after_1h"
    ]
]

## non-repeat 고객
non_repeat_sample_df = (
    customer_df[
        customer_df[
            "repeat_90d_after_1h"
        ] == 0
    ]
    .head(5)
)

## ineligible 고객
ineligible_sample_df = (
    customer_df[
        customer_df[
            "is_eligible_90d"
        ] == 0
    ]
    .head(5)
)

In [7]:
print("=== repeat 고객 ===")
print(
    repeat_sample_df[
        [
            "customer_unique_id",
            "first_approved_at",
            "second_approved_at_after_1h",
            "repeat_90d_after_1h"
        ]
    ]
    .to_string(
        index=False
    )
)

print("\n=== non-repeat 고객 ===")
print(
    non_repeat_sample_df[
        [
            "customer_unique_id",
            "first_approved_at",
            "second_approved_at_after_1h",
            "repeat_90d_after_1h"
        ]
    ]
    .to_string(
        index=False
    )
)

print("\n=== ineligible 고객 ===")
print(
    ineligible_sample_df[
        [
            "customer_unique_id",
            "first_approved_at",
            "second_approved_at_after_1h",
            "is_eligible_90d",
            "repeat_90d_after_1h"
        ]
    ]
    .to_string(
        index=False
    )
)

=== repeat 고객 ===
              customer_unique_id   first_approved_at second_approved_at_after_1h  repeat_90d_after_1h
0058f300f57d7b93c477a131a59b36c3 2018-02-19 17:20:52         2018-03-22 18:27:58                  1.0
00a39521eb40f7012db50455bf083460 2018-05-23 20:35:15         2018-06-03 10:50:00                  1.0
011575986092c30523ecb71ff10cb473 2018-02-17 16:06:43         2018-04-18 22:11:01                  1.0
012a218df8995d3ec3bb221828360c86 2018-05-08 20:15:40         2018-06-18 13:39:32                  1.0
013f4353d26bb05dc6652f1269458d8d 2017-11-24 15:53:40         2017-11-28 13:51:05                  1.0

=== non-repeat 고객 ===
              customer_unique_id   first_approved_at second_approved_at_after_1h  repeat_90d_after_1h
0000366f3b9a7992bf8c76cfdf3221e2 2018-05-10 11:11:18                         NaN                  0.0
0000b849f77a49e4a4ce2b2a4ca5be3f 2018-05-07 18:25:44                         NaN                  0.0
0000f46a3911fa3c0805444483337064 2017-03-